In [1]:
import polars


In [2]:
table = polars.read_excel(
    "https://cdn.bancentral.gov.do/documents/estadisticas/mercado-cambiario/documents/TASAS_CONVERTIBLES_OTRAS_MONEDAS.xlsx",
    read_options={
        "header_row": 2,

    })

Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 17, falling back to string


In [3]:
table.tail()

Año,Mes,Día,DOLAR AUSTRALIANO,REAL BRASILENO,DOLAR CANADIENSE,FRANCO SUIZO,YUAN CHINO,DERECHO ESPECIAL DE GIRO,CORONA DANESA,EURO,LIBRA ESTERLINA,YEN JAPONES,CORONA NORUEGA,LIBRA ESCOCESA,CORONA SUECA,DOLAR ESTADOUNIDENSE,*BOLIVAR FUERTE VENEZOLANO
i64,str,i64,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str
2026,"""Abr""",21,"""42.5759""","""11.9214""",43.4947,76.1715,"""8.7173""","""81.5612""",9.3547,69.9122,80.2651,0.3737,6.3748,80.2651,6.5038,59.4138,"""0.1242"""
2026,"""Abr""",22,"""42.5727""","""11.9339""",43.4894,76.0452,"""8.7046""","""81.4742""",9.3218,69.666,80.2705,0.373,6.3843,80.2705,6.4649,59.3761,"""0.124"""
2026,"""Abr""",23,"""42.3835""","""11.9315""",43.3647,75.4025,"""8.6684""","""81.0886""",9.2694,69.2708,79.883,0.3718,6.3476,79.883,6.4262,59.2362,"""0.1236"""
2026,"""Abr""",24,"""42.0387""","""11.7772""",43.1086,75.0342,"""8.6297""","""80.7843""",9.2288,69.0738,79.5657,0.3691,6.3067,79.5657,6.3839,58.9769,"""0.123"""
2026,"""Abr""",27,"""42.1181""","""11.8505""",43.0545,75.1052,"""8.6284""","""80.8312""",9.2591,69.1899,79.8519,0.3691,6.308,79.8519,6.384,58.89,"""0.1228"""


In [4]:
# set all not date columns to float
cols_to_convert = [col for col in table.columns if col not in ["Año", "Mes", "Día"]]
table = table.with_columns([polars.col(col).cast(polars.Float64) for col in cols_to_convert])

In [5]:
table.tail()

Año,Mes,Día,DOLAR AUSTRALIANO,REAL BRASILENO,DOLAR CANADIENSE,FRANCO SUIZO,YUAN CHINO,DERECHO ESPECIAL DE GIRO,CORONA DANESA,EURO,LIBRA ESTERLINA,YEN JAPONES,CORONA NORUEGA,LIBRA ESCOCESA,CORONA SUECA,DOLAR ESTADOUNIDENSE,*BOLIVAR FUERTE VENEZOLANO
i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026,"""Abr""",21,42.5759,11.9214,43.4947,76.1715,8.7173,81.5612,9.3547,69.9122,80.2651,0.3737,6.3748,80.2651,6.5038,59.4138,0.1242
2026,"""Abr""",22,42.5727,11.9339,43.4894,76.0452,8.7046,81.4742,9.3218,69.666,80.2705,0.373,6.3843,80.2705,6.4649,59.3761,0.124
2026,"""Abr""",23,42.3835,11.9315,43.3647,75.4025,8.6684,81.0886,9.2694,69.2708,79.883,0.3718,6.3476,79.883,6.4262,59.2362,0.1236
2026,"""Abr""",24,42.0387,11.7772,43.1086,75.0342,8.6297,80.7843,9.2288,69.0738,79.5657,0.3691,6.3067,79.5657,6.3839,58.9769,0.123
2026,"""Abr""",27,42.1181,11.8505,43.0545,75.1052,8.6284,80.8312,9.2591,69.1899,79.8519,0.3691,6.308,79.8519,6.384,58.89,0.1228


In [6]:
long_table = table.unpivot(
    index=["Año", "Mes", "Día"],
    variable_name="Moneda",
    value_name="rate_value",
)
long_table.shape



(83895, 5)

In [7]:
long_table.tail()

Año,Mes,Día,Moneda,rate_value
i64,str,i64,str,f64
2026,"""Abr""",21,"""*BOLIVAR FUERTE VENEZOLANO""",0.1242
2026,"""Abr""",22,"""*BOLIVAR FUERTE VENEZOLANO""",0.124
2026,"""Abr""",23,"""*BOLIVAR FUERTE VENEZOLANO""",0.1236
2026,"""Abr""",24,"""*BOLIVAR FUERTE VENEZOLANO""",0.123
2026,"""Abr""",27,"""*BOLIVAR FUERTE VENEZOLANO""",0.1228


In [9]:
long_table = long_table.filter(polars.col("rate_value").is_not_null()).sort(["Año", "Mes", "Día"])
long_table.shape

(72073, 5)

In [10]:
long_table.tail()

Año,Mes,Día,Moneda,rate_value
i64,str,i64,str,f64
2026,"""Mar""",31,"""YUAN CHINO""",8.7242
2026,"""Mar""",31,"""FRANCO SUIZO""",75.225
2026,"""Mar""",31,"""DOLAR CANADIENSE""",43.2681
2026,"""Mar""",31,"""CORONA SUECA""",6.3311
2026,"""Mar""",31,"""REAL BRASILENO""",11.5107
